# Chapter 12 — Cross-Validation and Model Evaluation Techniques

**Nama:** Omar Al Farizy Hutagalung  

**NIM:** 101032300034  

**Mata Kuliah:** Machine Learning  

**Tugas:** Tugas 2 — Code Reproduction & Theoretical Deep-Dive

---

## Objective

Chapter ini membahas train/test split, cross-validation, scoring metrics, perbandingan model, serta penggunaan evaluasi yang membantu mengurangi risiko estimasi performa yang bias.

## 1. Landasan Teori

Evaluasi model mengukur kemampuan generalisasi terhadap data yang tidak digunakan untuk melatih model.

- **Train/test split:** memisahkan data untuk pelatihan dan pengujian.
- **K-fold cross-validation:** membagi data menjadi K fold; setiap fold digunakan bergantian sebagai validation set.
- **Stratified K-fold:** menjaga proporsi kelas pada tugas klasifikasi.
- **Accuracy:** proporsi prediksi benar.
- **Precision, recall, F1:** metrik penting terutama saat distribusi kelas tidak seimbang.
- **Data leakage:** informasi dari data evaluasi masuk ke proses training sehingga estimasi performa menjadi terlalu optimistis.

Test set idealnya digunakan untuk evaluasi akhir setelah pemilihan model selesai.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, cross_val_score
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

## 2. Cross-validation dengan beberapa metrik

Semua preprocessing yang dipelajari dari data harus diletakkan dalam Pipeline agar di-fit secara terpisah pada setiap fold.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=42)
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, random_state=42
    ),
}

scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]
rows = []

for name, model in models.items():
    scores = cross_validate(
        model, X_train, y_train, cv=cv, scoring=scoring,
        return_train_score=False
    )
    row = {"Model": name}
    for metric in scoring:
        row[f"{metric}_mean"] = scores[f"test_{metric}"].mean()
        row[f"{metric}_std"] = scores[f"test_{metric}"].std()
    rows.append(row)

display(pd.DataFrame(rows).round(3))

## 3. Evaluasi akhir pada test set

Contoh berikut melatih model pada seluruh training set dan mengevaluasi test set yang belum digunakan selama cross-validation.

In [ ]:
final_model = models["Logistic Regression"]
final_model.fit(X_train, y_train)
test_pred = final_model.predict(X_test)
test_prob = final_model.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, test_pred))
print("Precision:", precision_score(y_test, test_pred))
print("Recall   :", recall_score(y_test, test_pred))
print("F1       :", f1_score(y_test, test_pred))
print(classification_report(y_test, test_pred))

ConfusionMatrixDisplay.from_predictions(y_test, test_pred)
plt.title("Confusion Matrix — Test Set")
plt.show()

## 4. Analisis dan Kesimpulan

- Cross-validation memberikan gambaran variasi performa antar-fold.
- Pemilihan metrik harus mengikuti tujuan dan biaya kesalahan aplikasi.
- Stratifikasi berguna pada klasifikasi dengan proporsi kelas berbeda.
- Preprocessing yang di-fit sebelum cross-validation dapat menyebabkan data leakage; gunakan Pipeline.
- Test set tidak seharusnya berulang kali digunakan untuk memilih hyperparameter.

**Kesimpulan:** evaluasi yang benar bukan hanya memilih satu angka tertinggi, tetapi memastikan proses validasi sesuai dengan struktur data dan tujuan model.